# Build a versioned inference service boundary

Use standard-library fixtures for request, startup, graph-signature and response contracts. The optional lab demonstrates the same boundaries with FastAPI and ONNX Runtime.

Fill in each starter, then run its check cell. Restart the kernel and run all cells before submitting. Keep your interpretation in the last Markdown cell.


## 1. Validate and order input features

Validate named features.

Implement parse_serving_request. Require exactly visits and tenure_days with strict integer values in [0,1000] and [0,36500]. Output them in that order. Reject invalid requests with ValueError.


In [ ]:
def parse_serving_request(payload):
    raise NotImplementedError


In [ ]:
assert parse_serving_request({"tenure_days":30,"visits":5}) == [5,30]
assert parse_serving_request({"visits":0,"tenure_days":36500}) == [0,36500]
assert parse_serving_request({"visits":1000,"tenure_days":0}) == [1000,0]
for bad in ([],{},{"visits":True,"tenure_days":30},{"visits":1001,"tenure_days":1},{"visits":2,"tenure_days":-1},{"visits":2,"tenure_days":1,"extra":0},{"visits":2,"tenure_days":True},{"visits":2.0,"tenure_days":1},{"visits":2,"tenure_days":36501}):
    try:
        parse_serving_request(bad)
    except ValueError:
        pass
    else:
        raise AssertionError("Reject malformed requests")
print("OK")


## 2. Separate readiness from liveness

Decide liveness and readiness.

Implement startup_status used by a deployment probe. Ready requires a live process, valid schema, and a nonmissing loaded digest equal to the expected digest.


In [ ]:
def startup_status(process_alive, loaded_digest, expected_digest, schema_valid):
    raise NotImplementedError


In [ ]:
assert startup_status(True,"x","x",True) == {"live":True,"ready":True}
assert startup_status(True,"x","y",True) == {"live":True,"ready":False}
assert startup_status(True,None,None,True) == {"live":True,"ready":False}
assert startup_status(False,"x","x",True) == {"live":False,"ready":False}
assert not startup_status(True,"x","x",False)["ready"]
print("OK")


## 3. Validate the exported input signature

Verify an exported shape.

Implement valid_tensor_contract. Require input name features, dtype float32 and rank two with positive batch size and exact feature width. Shape values are concrete integers.


In [ ]:
def valid_tensor_contract(name, dtype, shape, width):
    raise NotImplementedError


In [ ]:
assert valid_tensor_contract("features","float32",(1,2),2)
assert valid_tensor_contract("features","float32",(12,2),2)
assert not valid_tensor_contract("features","float64",(12,2),2)
assert not valid_tensor_contract("features","float32",(0,2),2)
assert not valid_tensor_contract("features","float32",(12,3),2)
assert not valid_tensor_contract("other","float32",(2,),2)
assert not valid_tensor_contract("features","float32",(2,),2)
assert not valid_tensor_contract("features","float32",(2,1,2),2)
print("OK")


## 4. Produce a versioned response

Return consistent response statuses.

Implement prediction_response for valid scores or None and a valid threshold. Return 503 with model_unavailable if not ready. Otherwise return 200, decision, score and model_version. Use review for missing score and urgent at or above threshold.


In [ ]:
def prediction_response(ready, score, threshold, version):
    raise NotImplementedError


In [ ]:
assert prediction_response(False,0.9,0.7,"v1") == (503,{"error":"model_unavailable"})
assert prediction_response(True,None,0.7,"v2") == (200,{"decision":"review","score":None,"model_version":"v2"})
assert prediction_response(True,0.7,0.7,"v2")[1]["decision"] == "urgent"
assert prediction_response(True,0.0,0.7,"v2")[1]["decision"] == "standard"
assert prediction_response(True,0.9,0.7,"v3") == (200,{"decision":"urgent","score":0.9,"model_version":"v3"})
print("OK")


## 5. Wire an injectable inference boundary

Implement serve_one(payload, ready, scorer, threshold, version). If unavailable return the
standard unavailable response without scoring. For a ready service, map ValueError from
parse_serving_request to (422, {"error":"invalid_input"}). Otherwise pass the ordered vector
to scorer and build the versioned response from its score or None. Do not catch scorer bugs
as client validation failures.


In [ ]:
def serve_one(payload, ready, scorer, threshold, version):
    raise NotImplementedError


In [ ]:
def unexpected(_):
    raise AssertionError("Invalid or unavailable requests must not reach inference")
assert serve_one({},False,unexpected,0.7,"v2") == (503,{"error":"model_unavailable"})
assert serve_one({},True,unexpected,0.7,"v2") == (422,{"error":"invalid_input"})
assert serve_one({"tenure_days":30,"visits":5},True,lambda row: 0.8 if row == [5,30] else None,0.7,"v2") == (200,{"decision":"urgent","score":0.8,"model_version":"v2"})
assert serve_one({"tenure_days":30,"visits":5},True,lambda _: None,0.7,"v2")[1]["decision"] == "review"
def broken(_):
    raise ValueError("scorer bug")
try:
    serve_one({"tenure_days":30,"visits":5},True,broken,0.7,"v3")
except ValueError:
    pass
else:
    raise AssertionError("Scorer failures are not client errors")
assert serve_one({"visits":5,"tenure_days":30},True,lambda _: 0.1,0.7,"v3") == (200,{"decision":"standard","score":0.1,"model_version":"v3"})
print("OK")


## Interpretation

Identify which checks run once at startup and which run on every request. Explain how to verify score and threshold parity before a runtime change.

**Your answer**


## Optional FastAPI, Docker and ONNX lab

Use the executable `serving_demo.py` in the ONNX input contract lesson and the Dockerfile
in the container lesson. Run the in-process tests, then build and start the local container.
Record the returned model version and one successful and one rejected input. Compare the
runtime's score and threshold outcome with the mathematical sigmoid on the same rows.
